In [1]:
from utils.analysis.valuation import BuySellSignalsAnalyzer, SignalsReporter, SectorAnalyzer
from utils.data import DataManager
from utils.tools.config import ANALYSIS_START_DATE, ANALYSIS_END_DATE


In [2]:
# 📊 CONFIGURACIÓN
# Las fechas vienen de config.py (TRADING_SIGNALS_CONFIG)
# Personaliza aquí solo si necesitas valores diferentes

# Tickers a analizar
TICKERS  = ["MSFT"]   # Agrega más: ["AAPL", "MSFT", "GOOGL", ...]

# Lookback para análisis de precios
LOOKBACK_YEARS = 5  # Años de histórico

# (Opcional) Fechas personalizadas - Deja vacío para usar config.py
USE_CUSTOM_DATES = False
CUSTOM_START = ""  # Ej: "2022-01-01"
CUSTOM_END = ""    # Ej: "2024-12-31"

print("📋 Configuración:")
print(f"   Tickers: {len(TICKERS)}")
print(f"   Lookback: {LOOKBACK_YEARS} años")
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    print(f"   Fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    print(f"   Fechas desde config.py: {ANALYSIS_START_DATE} → {ANALYSIS_END_DATE}")

📋 Configuración:
   Tickers: 1
   Lookback: 5 años
   Fechas desde config.py: 2020-01-01 → 2025-12-24


In [3]:
# 🔧 INICIALIZACIÓN
# Crear instancia compartida de DataManager
data_manager = DataManager()

# Crear analyzer con configuración
# Si USE_CUSTOM_DATES es True, usar fechas personalizadas
if USE_CUSTOM_DATES and CUSTOM_START and CUSTOM_END:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        start_date=CUSTOM_START,
        end_date=CUSTOM_END,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con fechas personalizadas: {CUSTOM_START} → {CUSTOM_END}")
else:
    analyzer = BuySellSignalsAnalyzer(
        data_manager=data_manager,
        lookback_years=LOOKBACK_YEARS
    )
    print(f"✅ Analyzer con configuración de config.py")

reporter = SignalsReporter()
print("✅ Reporter inicializado")

✅ Analyzer con configuración de config.py
✅ Reporter inicializado


In [4]:
# 🔍 ANÁLISIS DE SEÑALES
signals = []

print(f"\n🔍 Analizando {len(TICKERS)} ticker(s)...\n")

for ticker in TICKERS:
    try:
        # El analyzer ya tiene las fechas configuradas
        signal = analyzer.analyze_stock(ticker)
        signals.append(signal)
        
        # Mostrar resumen rápido
        emoji = "🟢" if signal.signal == "COMPRA" else "🔴" if signal.signal == "VENTA" else "🟡"
        print(f"{emoji} {ticker}: {signal.signal} "
              f"(Conf: {signal.confidence:.1f}%, Pot: {signal.upside_potential:+.1f}%)")
        
    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)[:60]}")

print(f"\n📊 Total analizado: {len(signals)}/{len(TICKERS)} tickers")


🔍 Analizando 1 ticker(s)...

Período: 2020-01-01 → 2026-01-29


[*********************100%***********************]  1 of 1 completed

🟡 MSFT: MANTENER (Conf: 50.0%, Pot: +27.2%)

📊 Total analizado: 1/1 tickers


In [5]:
# 📊 RESUMEN EN TABLA
if signals:
    summary_df = reporter.to_dataframe(signals)
    display(summary_df)
    
    print()
    reporter.print_summary(signals)
else:
    print("❌ No hay señales para mostrar")

,Ticker,Señal,Confianza,Valoración,Fundamental,Precio Actual,Precio Objetivo,Potencial
0,MSFT,MANTENER,50.0%,21.7,71.4,$482,$613,27.2%




RESUMEN DE SEÑALES

🟢 COMPRAS: 0

🔴 VENTAS: 0

🟡 MANTENER: 1


In [13]:
# 📊 COMPARATIVA VS SECTOR
sector_analyzer = SectorAnalyzer()

for ticker in TICKERS:
    result = sector_analyzer.analyze_vs_peers(ticker, fetch_peers=True)
    if not result.get('success'):
        print(f"❌ {ticker}: no se pudo analizar vs sector")
        continue
    sector = result.get('sector', 'N/A')
    peers = result.get('peers_analyzed', [])
    print(f"\n🏭 {ticker} | Sector: {sector} | Peers: {', '.join(peers) or 'ninguno'}")
    if result.get('peer_count', 0) > 0:
        display(result['comparison_df'])
        pct = result.get('percentiles', {})
        if isinstance(pct, dict) and 'percentile' in pct:
            print(f"   📈 Percentil vs sector: {pct['percentile']:.0f}% — {pct.get('interpretation', '')}")
        rel = result.get('relative_position', {})
        if isinstance(rel, dict) and 'valuation' in rel:
            v = rel['valuation']
            print(f"   💰 Valoración: {v.get('company_score', 0):.1f} vs media sector {v.get('peer_avg', 0):.1f} (diff: {v.get('vs_avg', 0):+.1f})")
    else:
        print("   ℹ️ Sin peers definidos para este sector en config (SECTOR_PEERS)")


🏭 MSFT | Sector: Technology | Peers: AAPL, GOOGL, GOOG, META, AMZN


,Ticker,Nombre,Sector,Rentabilidad,Salud,Crecimiento,Eficiencia,Valoración,Total,Conclusión
0,MSFT,Microsoft Corporation,Technology,81.942413,71.616439,58.687500,46.225974,21.754930,60.324348,REGULAR
1,AAPL,Apple Inc.,Technology,89.312031,58.420135,73.250000,56.878732,13.067137,62.074922,REGULAR
2,GOOGL,Alphabet Inc.,Communication Services,82.059572,79.192754,70.729167,38.433698,12.536376,62.104426,REGULAR
3,GOOG,Alphabet Inc.,Communication Services,82.059572,79.192754,70.729167,38.433698,12.548586,62.106257,REGULAR
4,META,"Meta Platforms, Inc.",Communication Services,77.316490,77.683832,38.500000,48.145516,22.674912,57.073145,REGULAR
5,AMZN,"Amazon.com, Inc.",Consumer Cyclical,53.307923,64.812449,69.333333,41.389960,21.410133,52.816773,REGULAR


   📈 Percentil vs sector: 33% — Por debajo del promedio
   💰 Valoración: 21.8 vs media sector 16.4 (diff: +5.3)


In [10]:
# 🏆 TOP OPORTUNIDADES DE COMPRA
if signals:
    reporter.print_top_opportunities(signals, top_n=5)
else:
    print("❌ No hay señales para analizar")


⚠️ No hay oportunidades de compra identificadas


In [11]:
# 🟢 DETALLE DE SEÑALES DE COMPRA
compras = [s for s in signals if s.signal == "COMPRA"]

if compras:
    # Ordenar por confianza
    top_compras = sorted(compras, key=lambda x: x.confidence, reverse=True)
    
    print(f"📈 Encontradas {len(compras)} señales de COMPRA\n")
    print("=" * 65)
    
    for signal in top_compras:
        reporter.print_signal(signal)
        print()
else:
    print("ℹ️  No hay señales de COMPRA en este momento")

ℹ️  No hay señales de COMPRA en este momento


In [12]:
# 🏭 ANÁLISIS POR SECTOR
# Analiza múltiples tickers de un sector específico

TECH_SECTOR  = ["MSFT", "META", "NVDA", "GOOGL", "AMZN", "IBM"]

print("🔍 Analizando sector tecnológico...\n")
tech_signals = []

for ticker in TECH_SECTOR:
    try:
        signal = analyzer.analyze_stock(ticker)
        tech_signals.append(signal)
    except Exception as e:
        print(f"⚠️  {ticker}: {str(e)[:40]}")

if tech_signals:
    # Mostrar solo las mejores oportunidades
    compras_tech = [s for s in tech_signals if s.signal == "COMPRA"]
    
    if compras_tech:
        print(f"\n✅ {len(compras_tech)} oportunidades de compra en Tech:\n")
        
        tech_df = reporter.to_dataframe(compras_tech)
        display(tech_df.sort_values('Confianza', ascending=False))
    else:
        print("\nℹ️  No hay señales de compra en el sector tech actualmente")

🔍 Analizando sector tecnológico...



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2026-01-29



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2026-01-29


...

Período: 2020-01-01 → 2026-01-29



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2026-01-29



[*********************100%***********************]  1 of 1 completed

Período: 2020-01-01 → 2026-01-29

ℹ️  No hay señales de compra en el sector tech actualmente
